In [ ]:
from typing import Final
from pathlib import Path
from dotenv import find_dotenv
import numpy as np
from geopandas import GeoDataFrame
import matplotlib.pyplot as plt

## Inspect tiff as image

In [ ]:
PROJECT_DIR: Final[Path] = Path(find_dotenv(".env", 1, 1)).absolute().parent

In [ ]:
import rasterio as r
with r.open(PROJECT_DIR.joinpath("data/cardiff-monm-037se-2.tif"), "r") as src:
    data = src.read()

Image is binary (black/white):

In [ ]:
np.unique(data)

In [ ]:
fig, ax = plt.subplots(figsize = (20, 20 * data.shape[1] / data.shape[2]))
ax.imshow(data[0], cmap = "gray_r")

In [ ]:
# from PIL import Image
# img = Image.open(PROJECT_DIR.joinpath("data/cardiff-monm-037se-2.tif"))
# np.array(img)

# plt.imshow(np.array(img), cmap = "gray")

## Parse .tab file to obtain metadata for georeferencing

In [ ]:
from edina import parse_edina_tab_file
metadata =\
    parse_edina_tab_file(PROJECT_DIR.joinpath("data/cardiff-monm-037se-2.tab"))
metadata.crs # assess returned GeoDataFrame crs

In [ ]:
# Assess geometric features
check = metadata.loc[0, "geometry"]
print(f"Geometry: {check}\nNorthing: {check.y}\nEasting: {check.x}")

## Code to convert coordinates to image pixel locations

In [ ]:
from edina import get_transformer_from_geodataframe
geo_transformer = get_transformer_from_geodataframe(metadata)
geo_transformer.rowcol(check.x, check.y)

Check example points from GB1900 gazetteer are plotted in the correct place (around the C of cardiff)

In [ ]:
from json import load as load_json
from shapely import Point

with open(PROJECT_DIR.joinpath("data/gb1900-example.json"), "r") as src:
    gb1900 = load_json(src)

for record in gb1900["data"]:
    record["geometry"] = Point(record.pop("Lon-Lat"))
# gb1900_pixels = [geo_transformer.rowcol(*el["Lon-Lat"]) for el in gb1900["data"]]

gb1900 = GeoDataFrame\
    .from_records(gb1900["data"]).set_crs(gb1900["crs"]).to_crs("EPSG:27700")

gb1900_pixels = geo_transformer.rowcol(gb1900.geometry.x, gb1900.geometry.y)

ax.plot(*gb1900_pixels[::-1], "or")
fig